# Image Caption Generator — Colab Training Notebook

Runs the full pipeline from `../` (ResNet-50 encoder + Bahdanau attention + LSTM decoder) against the real Flickr8k dataset, with GPU acceleration.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

Steps in this notebook:
1. Clone the repo (or upload the folder) and install dependencies
2. Download Flickr8k (Hugging Face mirror, no API key needed)
3. Build `TRAIN_PAIRS` / `VAL_PAIRS` / `TEST_PAIRS`
4. Train
5. Evaluate (BLEU / METEOR / CIDEr)
6. Save `best_checkpoint.pth` to Google Drive so it survives the session

## 1. Get the code onto Colab

In [ ]:
# Option A: clone from GitHub (recommended -- push this project to a repo first)
# !git clone https://github.com/<your-username>/image-caption-generator.git
# %cd image-caption-generator

# Option B: upload the project as a zip via the Colab file browser, then:
# !unzip -q image-caption-generator.zip
# %cd image-caption-generator

!pwd
!ls

In [ ]:
!pip install -q -e .

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable a GPU runtime!)")

## 2. Download Flickr8k (Hugging Face mirror)

In [ ]:
!pip install -q datasets
from datasets import load_dataset
import os

ds = load_dataset("jxie/flickr8k", split="train")
print(ds)
print(ds[0].keys())

In [ ]:
# The jxie/flickr8k mirror stores one row per image with 5 caption columns
# (caption_0..caption_4). If you use a different mirror, check the printed
# keys above and adjust the loop below accordingly.

os.makedirs("data/flickr8k/Images", exist_ok=True)
rows = []
caption_cols = [c for c in ds.column_names if c.startswith("caption")]

for i, example in enumerate(ds):
    fname = f"img_{i}.jpg"
    example["image"].convert("RGB").save(f"data/flickr8k/Images/{fname}")
    for col in caption_cols:
        cap = example[col]
        if cap:
            rows.append((fname, cap.strip()))

print(f"Saved {i + 1} images, {len(rows)} (image, caption) pairs")

## 3. Train / val / test split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame(rows, columns=["image", "caption"])
unique_images = df["image"].unique()

train_imgs, temp_imgs = train_test_split(unique_images, test_size=0.2, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

def pairs_for(image_list):
    subset = df[df["image"].isin(image_list)]
    return list(zip(subset["image"], subset["caption"]))

TRAIN_PAIRS = pairs_for(train_imgs)
VAL_PAIRS = pairs_for(val_imgs)
TEST_PAIRS = pairs_for(test_imgs)

print(f"train: {len(TRAIN_PAIRS)} pairs / {len(train_imgs)} images")
print(f"val:   {len(VAL_PAIRS)} pairs / {len(val_imgs)} images")
print(f"test:  {len(TEST_PAIRS)} pairs / {len(test_imgs)} images")

## 4. Train

This mirrors `train.py`'s `main()` but runs inline so `TRAIN_PAIRS`/`VAL_PAIRS` from above are used directly, instead of editing the script.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader

from caption_generator.data.dataset import Flickr8kDataset, collate_fn
from caption_generator.data.vocabulary import Vocabulary
from caption_generator.models.decoder import DecoderWithAttention
from caption_generator.models.encoder import EncoderCNN
from caption_generator.train import train_one_epoch, validate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMAGE_DIR = "data/flickr8k/Images"
TRAIN_CAPTIONS_RAW = [cap for _, cap in TRAIN_PAIRS]

vocab = Vocabulary(min_word_freq=5).build(TRAIN_CAPTIONS_RAW)
pad_idx = vocab.word2idx[vocab.PAD_TOKEN]
print(f"Vocab size: {len(vocab)}")

train_dataset = Flickr8kDataset(IMAGE_DIR, TRAIN_PAIRS, vocab, split="train")
val_dataset = Flickr8kDataset(IMAGE_DIR, VAL_PAIRS, vocab, split="val")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                           collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                         collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)

encoder = EncoderCNN(fine_tune=False).to(device)
decoder = DecoderWithAttention(vocab_size=len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(decoder.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

NUM_EPOCHS = 15  # early stopping below decides the real stopping point
EARLY_STOP_PATIENCE = 4
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion, device, pad_idx)
    val_loss = validate(encoder, decoder, val_loader, criterion, device)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  lr={current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save({
            "encoder_state": encoder.state_dict(),
            "decoder_state": decoder.state_dict(),
            "vocab_word2idx": vocab.word2idx,
            "vocab_idx2word": vocab.idx2word,
        }, "best_checkpoint.pth")
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"No val_loss improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early.")
            break

## 5. Evaluate on the test split (BLEU / METEOR / CIDEr)

In [ ]:
!pip install -q pycocoevalcap

from collections import defaultdict

from caption_generator.evaluate import evaluate

test_pairs_by_image = defaultdict(list)
for fname, cap in TEST_PAIRS:
    test_pairs_by_image[fname].append(cap)

scores = evaluate("best_checkpoint.pth", dict(test_pairs_by_image), IMAGE_DIR, device=device)
for metric, value in scores.items():
    print(f"{metric}: {value:.4f}")

## 6. Attention visualizations (a few examples)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image

from caption_generator.models.caption_model import CaptionModel

checkpoint = torch.load("best_checkpoint.pth", map_location=device)
vocab = Vocabulary()
vocab.word2idx = checkpoint["vocab_word2idx"]
vocab.idx2word = checkpoint["vocab_idx2word"]

encoder = EncoderCNN(fine_tune=False)
encoder.load_state_dict(checkpoint["encoder_state"])
decoder = DecoderWithAttention(vocab_size=len(vocab))
decoder.load_state_dict(checkpoint["decoder_state"])
model = CaptionModel(encoder, decoder, vocab, device=device)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

sample_fname = TEST_PAIRS[0][0]
image = Image.open(f"{IMAGE_DIR}/{sample_fname}").convert("RGB")
image_tensor = transform(image).unsqueeze(0)

caption, alphas = model.generate_greedy(image_tensor)
print(f"Generated caption: {caption}")

plt.imshow(image)
plt.title(caption)
plt.axis("off")
plt.show()

## 7. Save the checkpoint to Google Drive (so it survives the Colab session ending)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/image-caption-generator
!cp best_checkpoint.pth /content/drive/MyDrive/image-caption-generator/
print("Saved to Google Drive: MyDrive/image-caption-generator/best_checkpoint.pth")